In [1]:
!pip install pandas numpy streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.1 MB/s eta 0:00:00


In [2]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

In [3]:
conn = sqlite3.connect("bank.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    account TEXT,
    type TEXT,
    amount REAL,
    timestamp TEXT
)
""")

conn.commit()

In [4]:
def detect_fraud(account, amount):

    df = pd.read_sql_query(
        f"SELECT amount FROM transactions WHERE account='{account}' AND type='withdraw'",
        conn
    )

    if len(df) < 5:
        return False

    mean = df["amount"].mean()
    std = df["amount"].std()

    z_score = (amount - mean) / std if std != 0 else 0

    return abs(z_score) > 3

In [5]:
class BankAccount:

    def __init__(self, name, balance=0):
        self.name = name
        self.__balance = balance

    def deposit(self, amount):

        self.__balance += amount

        cursor.execute(
            "INSERT INTO transactions(account,type,amount,timestamp) VALUES (?,?,?,?)",
            (self.name,"deposit",amount,str(datetime.now()))
        )

        conn.commit()

    def withdraw(self, amount):

        if detect_fraud(self.name, amount):
            print("⚠ Fraud Alert: Unusual withdrawal detected")

        if amount > self.__balance:
            print("Insufficient balance")
            return

        self.__balance -= amount

        cursor.execute(
            "INSERT INTO transactions(account,type,amount,timestamp) VALUES (?,?,?,?)",
            (self.name,"withdraw",amount,str(datetime.now()))
        )

        conn.commit()

    def get_balance(self):
        return self.__balance

In [6]:
class SavingsAccount(BankAccount):

    def add_interest(self, rate=0.04):
        interest = self.get_balance() * rate
        self.deposit(interest)


class CurrentAccount(BankAccount):

    def __init__(self, name, balance=0, overdraft=5000):
        super().__init__(name, balance)
        self.overdraft = overdraft

In [7]:
%%writefile app.py

import streamlit as st
import sqlite3
import pandas as pd

conn = sqlite3.connect("bank.db")

st.title("Mini Banking Dashboard")

account = st.text_input("Account Name")

action = st.selectbox("Action",["Deposit","Withdraw","View Transactions"])

amount = st.number_input("Amount",0)

if st.button("Submit"):

    if action == "Deposit":

        conn.execute(
            "INSERT INTO transactions(account,type,amount,timestamp) VALUES (?,?,?,datetime('now'))",
            (account,"deposit",amount)
        )
        conn.commit()

        st.success("Deposit Successful")

    elif action == "Withdraw":

        conn.execute(
            "INSERT INTO transactions(account,type,amount,timestamp) VALUES (?,?,?,datetime('now'))",
            (account,"withdraw",amount)
        )
        conn.commit()

        st.success("Withdrawal Successful")


if action == "View Transactions":

    df = pd.read_sql_query(
        f"SELECT * FROM transactions WHERE account='{account}'",
        conn
    )

    st.dataframe(df)

    if not df.empty:
        st.line_chart(df["amount"])

Writing app.py


In [8]:
from pyngrok import ngrok
import os

public_url = ngrok.connect(8501)
print(public_url)

os.system("streamlit run app.py &")

ERROR:pyngrok.process.ngrok:t=2026-03-16T11:24:15+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-16T11:24:15+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.